In [1]:
from google.colab import drive
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, recall_score, f1_score

import time

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
train_path = '/content/drive/MyDrive/datasets_big_data/KDD/train_500000.csv'
test_path = '/content/drive/MyDrive/datasets_big_data/KDD/validation.csv'
manifest_val_path = '/content/drive/MyDrive/datasets_big_data/KDD/validation.manifest.csv'

df_train = pd.read_csv(train_path, header=None)
df_test = pd.read_csv(test_path, header=None)
manifest_val = pd.read_csv(manifest_val_path)

print(f"entrenamiento: {df_train.shape}")
print(f"prueba: {df_test.shape}")
print(f"Filas en validation.csv: {len(df_test)}")
print(f"Filas en manifest_val: {len(manifest_val)}")

entrenamiento: (500000, 42)
prueba: (734765, 42)
Filas en validation.csv: 734765
Filas en manifest_val: 734765


In [3]:
X_train = df_train.iloc[:, :-1].copy()
y_train_raw = df_train.iloc[:, -1]

X_val = df_test.iloc[:, :-1].copy()

print(len(X_val))
print(len(manifest_val))

# codificacion de etiquetads
y_train = np.where(y_train_raw.str.contains('normal'), 0, 1)
y_val = np.where(manifest_val['source_label'].str.contains('normal'), 0, 1)

#variables Categóricas (filass 1, 2 y 3)
categorical_cols = [1, 2, 3]

# preproc HistGradientBoosting
#maneja categóricas de forma nativa
for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_val[col] = X_val[col].astype('category')

#aplicar OrdinalEncoder.
# más eficiente en memoria que One-Hot Encoding para 500k registros.
# encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
# X_train_encoded = X_train.copy()
# X_val_encoded = X_val.copy()

# X_train_encoded[categorical_cols] = encoder.fit_transform(X_train_encoded[categorical_cols].astype(str))
# X_val_encoded[categorical_cols] = encoder.transform(X_val_encoded[categorical_cols].astype(str))
member_name = "Caleb"
dataset_name = "KDD"
sample_size = "500k"

resultados_validacion = []

734765
734765


# HistGradientBoostingClassifier

---



In [4]:
from sklearn.ensemble import HistGradientBoostingClassifier

print("--- Entrenando HistGradientBoosting ---")
start_time = time.time()

# Se saebe que las columnas 1, 2, 3 son categoricas
hgb_model = HistGradientBoostingClassifier(
    categorical_features=categorical_cols,
    random_state=10,
    max_iter=200
)

# entrenamiento
hgb_model.fit(X_train, y_train)
hgb_train_time = time.time() - start_time
print(f"Tiempo de entrenamiento: {hgb_train_time:.2f} segundos")

# Predicción y Evaluación
y_pred_hgb = hgb_model.predict(X_val)

print("\nResultados HistGradientBoosting:")
print(f"Exactitud (Accuracy): {accuracy_score(y_val, y_pred_hgb):.4f}")
print("\nMatriz de Confusión:")
print(confusion_matrix(y_val, y_pred_hgb))
print("\nReporte de Clasificación:")
print(classification_report(y_val, y_pred_hgb, target_names=['Normal (0)', 'Ataque (1)']))


resultados_validacion.append({
    'member': member_name,
    'dataset': dataset_name,
    'sample': sample_size,
    'model': 'HistGradientBoosting',
    'accuracy': accuracy_score(y_val, y_pred_hgb),
    'recall': recall_score(y_val, y_pred_hgb),
    'f1': f1_score(y_val, y_pred_hgb),
    'training_time_seconds': hgb_train_time
})

--- Entrenando HistGradientBoosting ---
Tiempo de entrenamiento: 24.95 segundos

Resultados HistGradientBoosting:
Exactitud (Accuracy): 0.9994

Matriz de Confusión:
[[145495    422]
 [    25 588823]]

Reporte de Clasificación:
              precision    recall  f1-score   support

  Normal (0)       1.00      1.00      1.00    145917
  Ataque (1)       1.00      1.00      1.00    588848

    accuracy                           1.00    734765
   macro avg       1.00      1.00      1.00    734765
weighted avg       1.00      1.00      1.00    734765



# Random Forest

---



In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder

print("--- Entrenando Random Forest ---")
start_time = time.time()

# 1. Codificación de variables categóricas
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train_encoded = X_train.copy()
X_val_encoded = X_val.copy()

X_train_encoded[categorical_cols] = encoder.fit_transform(X_train_encoded[categorical_cols].astype(str))
X_val_encoded[categorical_cols] = encoder.transform(X_val_encoded[categorical_cols].astype(str))

# 2. Inicialización y Entrenamiento
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1 # Usa todos los núcleos disponibles
)

rf_model.fit(X_train_encoded, y_train)
rf_train_time = time.time() - start_time
print(f"Tiempo de entrenamiento: {rf_train_time:.2f} segundos")

# 3. Predicción y Evaluación
y_pred_rf = rf_model.predict(X_val_encoded)

print("\nResultados Random Forest:")
print(f"Exactitud (Accuracy): {accuracy_score(y_val, y_pred_rf):.4f}")
print("\nMatriz de Confusión:")
print(confusion_matrix(y_val, y_pred_rf))
print("\nReporte de Clasificación:")
print(classification_report(y_val, y_pred_rf, target_names=['Normal (0)', 'Ataque (1)']))


resultados_validacion.append({
    'member': member_name,
    'dataset': dataset_name,
    'sample': sample_size,
    'model': 'Random Forest',
    'accuracy': accuracy_score(y_val, y_pred_rf),
    'recall': recall_score(y_val, y_pred_rf),
    'f1': f1_score(y_val, y_pred_rf),
    'training_time_seconds': rf_train_time
})

--- Entrenando Random Forest ---
Tiempo de entrenamiento: 20.78 segundos

Resultados Random Forest:
Exactitud (Accuracy): 0.9999

Matriz de Confusión:
[[145907     10]
 [    30 588818]]

Reporte de Clasificación:
              precision    recall  f1-score   support

  Normal (0)       1.00      1.00      1.00    145917
  Ataque (1)       1.00      1.00      1.00    588848

    accuracy                           1.00    734765
   macro avg       1.00      1.00      1.00    734765
weighted avg       1.00      1.00      1.00    734765



# Extra Trees

---



In [10]:
from sklearn.ensemble import ExtraTreesClassifier

print("--- Entrenando Extra Trees ---")
start_time = time.time()

# Inicialización y Entrenamiento (usando los datos codificados del bloque anterior)
et_model = ExtraTreesClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1 # Usa todos los núcleos disponibles
)

et_model.fit(X_train_encoded, y_train)
et_train_time = time.time() - start_time
print(f"Tiempo de entrenamiento: {et_train_time:.2f} segundos")

# Predicción y Evaluación
y_pred_et = et_model.predict(X_val_encoded)

print("\nResultados Extra Trees:")
print(f"Exactitud (Accuracy): {accuracy_score(y_val, y_pred_et):.4f}")
print("\nMatriz de Confusión:")
print(confusion_matrix(y_val, y_pred_et))
print("\nReporte de Clasificación:")
print(classification_report(y_val, y_pred_et, target_names=['Normal (0)', 'Ataque (1)']))


resultados_validacion.append({
    'member': member_name,
    'dataset': dataset_name,
    'sample': sample_size,
    'model': 'Extra Trees',
    'accuracy': accuracy_score(y_val, y_pred_rf),
    'recall': recall_score(y_val, y_pred_rf),
    'f1': f1_score(y_val, y_pred_rf),
    'training_time_seconds': rf_train_time
})


df_resultados = pd.DataFrame(resultados_validacion)



--- Entrenando Extra Trees ---
Tiempo de entrenamiento: 16.43 segundos

Resultados Extra Trees:
Exactitud (Accuracy): 0.9993

Matriz de Confusión:
[[145466    451]
 [    33 588815]]

Reporte de Clasificación:
              precision    recall  f1-score   support

  Normal (0)       1.00      1.00      1.00    145917
  Ataque (1)       1.00      1.00      1.00    588848

    accuracy                           1.00    734765
   macro avg       1.00      1.00      1.00    734765
weighted avg       1.00      1.00      1.00    734765



In [12]:
# print(df_resultados)

predicciones_finales = y_pred_rf
df_validation_predictions = pd.DataFrame({
    'observation_id': manifest_val['observation_id'],
    'prediction': predicciones_finales
})

df_validation_predictions['observation_id'] = df_validation_predictions['observation_id'].str.split(':').str[1]
# df_validation_predictions['observation_id'] = pd.to_numeric(df_validation_predictions['observation_id'])

path_final = '/content/drive/MyDrive/datasets_big_data/KDD/validation_predictions.csv'
df_validation_predictions.to_csv(path_final, index=False)
